# Week 1 — Task 3: Distributions and Relationships

**Name:** Tanzeela Awan  
**Roll No:** SU92-BSDSM-S23-002  

**Dataset:** `penguins.csv` — 344 penguins, 3 species, 3 islands.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('data/penguins.csv')
print(f'Shape: {df.shape}')
print('\nMissing values:')
print(df.isnull().sum())

## Handling Missing Values

The dataset has:
- 2 penguins missing **all four** measurements — cannot be used for numeric charts.
- 11 penguins missing **sex** — can still be used when sex is not needed.

**Decision:** Drop the 2 rows missing all measurements for all charts. Keep the 11 missing sex for charts that do not use sex, and drop them only for the sex-split chart. This is stated on each affected chart.

In [ ]:
measurement_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
df_clean = df.dropna(subset=measurement_cols).copy()
print(f'Retained {len(df_clean)} of {len(df)} penguins (dropped {len(df)-len(df_clean)} missing all measurements)')
print(f'Species distribution:\n{df_clean["species"].value_counts()}')

## Step 1 — Distribution of Body Mass (All Penguins)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df_clean['body_mass_g'], bins=30, color='#2166ac', edgecolor='white', alpha=0.85)
ax.set_title(
    'Body mass shows two humps — a hint that multiple groups are mixed together\n'
    '(n=342; 2 penguins with missing measurements excluded)',
    loc='left'
)
ax.set_xlabel('Body mass (g)')
ax.set_ylabel('Count')
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task3_body_mass_all.png', dpi=150, bbox_inches='tight')
plt.show()
print('Shape: bimodal — two humps around ~3500 g and ~5000 g suggest two overlapping groups.')

## Step 2 — Body Mass Split by Species

In [ ]:
species_colors = {'Adelie': '#d73027', 'Chinstrap': '#fc8d59', 'Gentoo': '#4575b4'}
species_list = sorted(df_clean['species'].unique())

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for i, sp_name in enumerate(species_list):
    sub = df_clean[df_clean['species'] == sp_name]['body_mass_g']
    axes[i].hist(sub, bins=20, color=species_colors[sp_name], edgecolor='white', alpha=0.85)
    axes[i].set_title(f'{sp_name}\n(n={len(sub)})', fontsize=11)
    axes[i].set_xlabel('Body mass (g)')
    for spine in ('top', 'right'):
        axes[i].spines[spine].set_visible(False)

axes[0].set_ylabel('Count')
fig.suptitle(
    'Each species is unimodal — the two-hump shape arises from mixing Adelie/Chinstrap with the heavier Gentoo',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig('task3_body_mass_species.png', dpi=150, bbox_inches='tight')
plt.show()

**Does splitting by species explain the bimodal shape?**

Yes. Adelie and Chinstrap penguins cluster around 3,500–3,750 g (forming the left hump), while Gentoo penguins average around 5,000–5,500 g (forming the right hump). Each species individually shows a roughly bell-shaped (unimodal) distribution. The bimodality in the combined chart is entirely explained by the mixture of species — it is a **Simpson's paradox at the distributional level**.

## Step 3 — Simpson's Paradox: Bill Length vs Bill Depth

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Per-species scatter and regression
for sp_name, grp in df_clean.groupby('species'):
    ax.scatter(grp['bill_length_mm'], grp['bill_depth_mm'],
               color=species_colors[sp_name], label=sp_name, s=40, alpha=0.7, edgecolors='none')
    m, b = np.polyfit(grp['bill_length_mm'], grp['bill_depth_mm'], 1)
    xs = np.linspace(grp['bill_length_mm'].min(), grp['bill_length_mm'].max(), 80)
    ax.plot(xs, m * xs + b, color=species_colors[sp_name], lw=2)

# Overall regression line
m_all, b_all = np.polyfit(df_clean['bill_length_mm'], df_clean['bill_depth_mm'], 1)
xs_all = np.linspace(df_clean['bill_length_mm'].min(), df_clean['bill_length_mm'].max(), 100)
ax.plot(xs_all, m_all * xs_all + b_all, color='black', lw=2, linestyle='--',
        label='Overall trend (negative)')

r_all = df_clean['bill_length_mm'].corr(df_clean['bill_depth_mm'])
ax.set_title(
    "Simpson's Paradox: the overall bill correlation is negative (r={:.2f}),\n"
    "but within every species it is positive".format(r_all),
    loc='left'
)
ax.set_xlabel('Bill length (mm)')
ax.set_ylabel('Bill depth (mm)')
ax.legend(frameon=False)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task3_simpsons_paradox.png', dpi=150, bbox_inches='tight')
plt.show()

print('Per-species correlations:')
for sp_name, grp in df_clean.groupby('species'):
    r = grp['bill_length_mm'].corr(grp['bill_depth_mm'])
    print(f'  {sp_name}: r = {r:.3f}')

### Explaining Simpson's Paradox

Across *all* penguins the overall trend is **negative** — longer bills go with shallower depths. Within *each* species the trend is **positive** — longer bills also have deeper depths.

The paradox arises because the species occupy different regions of the chart. Gentoo are large-bodied with long bills but relatively shallow depths; Adelie are smaller with shorter but proportionally deeper bills. **Species is a confounder** — it is correlated with both bill length and bill depth, and when ignored it reverses the apparent sign of the relationship.

The lesson (look up 'Simpson's paradox'): a correlation measured across groups can be opposite in direction from the correlation measured within each group. Never report a correlation without first checking it within your natural subgroups.

## Step 4 — Average Body Mass by Species and Sex

In [ ]:
df_sex = df_clean.dropna(subset=['sex'])
excluded_sex = len(df_clean) - len(df_sex)

mass_sex = (
    df_sex.groupby(['species', 'sex'])['body_mass_g']
    .mean()
    .reset_index()
)
mass_sex['body_mass_kg'] = mass_sex['body_mass_g'] / 1000

species_list = sorted(mass_sex['species'].unique())
x = np.arange(len(species_list))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 6))

males   = mass_sex[mass_sex['sex'] == 'Male'].set_index('species').reindex(species_list)
females = mass_sex[mass_sex['sex'] == 'Female'].set_index('species').reindex(species_list)

ax.bar(x + width / 2, males['body_mass_kg'],   width, label='Male',   color='#4575b4', alpha=0.85)
ax.bar(x - width / 2, females['body_mass_kg'], width, label='Female', color='#d73027', alpha=0.85)

for xi, m_val, f_val in zip(x, males['body_mass_kg'], females['body_mass_kg']):
    ax.text(xi + width / 2, m_val + 0.04, f'{m_val:.2f}', ha='center', fontsize=8)
    ax.text(xi - width / 2, f_val + 0.04, f'{f_val:.2f}', ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(species_list)
ax.set_title(
    f'Males are heavier in all three species; Gentoo stands out overall\n'
    f'(n={len(df_sex)}; {excluded_sex} penguins with missing sex excluded)',
    loc='left'
)
ax.set_ylabel('Average body mass (kg)')
ax.set_ylim(0, 6.5)
ax.legend(frameon=False)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task3_body_mass_sex.png', dpi=150, bbox_inches='tight')
plt.show()

**Chart-type justification:** A grouped bar chart. The two categorical variables (species and sex) form natural groups, and the quantity being compared (average body mass) is continuous. The within-group gap between bars is smaller than the between-group gap, using **proximity** to signal that male/female bars within a species belong together. A table would hide the magnitude of the sex difference; a scatter plot would be misleading for a grouped mean.